# PyTorch Model Saving, Loading, and Inference

**PyTorch Interview Preparation — Notebook 13**  
**Study time:** 3–4 hours

Training a model is not the finish line. A practical PyTorch engineer must be able to save it, restore it on another device, resume training exactly, and turn raw outputs into reliable predictions.

> 🔥 **Interview Essential:** `state_dict`, `torch.save`, `torch.load`, architecture-compatible loading, weights-only vs full checkpoints, `map_location`, `model.eval()`, `torch.inference_mode()`, resuming training, best-model saving, and preprocessing consistency.

Difficulty labels: 🟢 Easy · 🟡 Medium · 🔴 Hard/optional


## 1. Setup


In [ ]:
import os
import tempfile
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

tmpdir = tempfile.mkdtemp(prefix="pytorch_save_load_")
print("Device:", device)
print("Temporary artifacts:", tmpdir)


## 2. Why Saving Models Matters

Training may take minutes, hours, days, or weeks. You should not retrain whenever you need a prediction.

```text
train → save → close program → later → load → inference or resume training
```

🔥 **Interview Essential**

- **Goal A — inference:** save trained weights; rebuild the architecture and load them later.
- **Goal B — resume:** save a checkpoint containing model state plus optimizer, epoch/step, and any other state that changes future training.


## 3. What Is a `state_dict`? 🔥 Interview Essential


In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=32, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


model = SimpleClassifier()
state = model.state_dict()
for key, value in state.items():
    print(f"{key:20s} {tuple(value.shape)}")


A `state_dict` is an ordered Python mapping from parameter/buffer names to tensors. The key identifies state such as `net.0.weight`; the tensor is its value.

> 🔥 **Very common interview point:** a state dict contains parameter and persistent-buffer values—not the Python class definition. A usable model is **architecture code + compatible state dict**.


## 4. Inspecting `model.state_dict()`


In [ ]:
assert list(state) == ["net.0.weight", "net.0.bias", "net.2.weight", "net.2.bias"]
print("State entries:", len(state))
print("Model training mode by default:", model.training)


## 5. Saving Model Weights 🟢


In [ ]:
weights_path = os.path.join(tmpdir, "model_weights.pt")
torch.save(model.state_dict(), weights_path)
print(weights_path, os.path.getsize(weights_path), "bytes")


`.pt` and `.pth` are naming conventions; they have no functional difference.


### Exercise 1 — Save weights


In [ ]:
exercise_weights_path = os.path.join(tmpdir, "exercise_weights.pt")
# Your turn: save only model weights. Reference solution:
torch.save(model.state_dict(), exercise_weights_path)

assert os.path.exists(exercise_weights_path)
loaded_exercise = torch.load(exercise_weights_path, map_location="cpu", weights_only=True)
assert isinstance(loaded_exercise, dict)
print("✅ Correct!")


## 6. Loading Model Weights 🔥 Interview Essential


In [ ]:
new_model = SimpleClassifier()  # architecture must exist first
loaded_state = torch.load(weights_path, map_location="cpu", weights_only=True)
load_result = new_model.load_state_dict(loaded_state)
new_model.eval()
print(load_result)


The reliable order is: **re-create architecture → load parameter dictionary → move to target device → call `eval()` for inference**.


## 7. Architecture + Weights


`state_dict` loading copies values into matching named parameters and buffers. Configuration metadata such as `input_dim`, `hidden_dim`, and `num_classes` reduces reconstruction mistakes, but the model class still needs to exist.


## 8. Verify the Loaded Model


In [ ]:
model.eval()
new_model.eval()
X_verify = torch.randn(8, 10)
with torch.inference_mode():
    output1 = model(X_verify)
    output2 = new_model(X_verify)

assert torch.allclose(output1, output2)
for (name1, p1), (name2, p2) in zip(model.named_parameters(), new_model.named_parameters()):
    assert name1 == name2 and torch.equal(p1, p2)
print("✅ Outputs and parameters match exactly")


### Exercise 2 — Central save/load round trip


In [ ]:
torch.manual_seed(7)
round_model = SimpleClassifier()
round_opt = optim.AdamW(round_model.parameters(), lr=1e-3)
round_x, round_y = torch.randn(16, 10), torch.randint(0, 3, (16,))
round_loss = nn.CrossEntropyLoss()(round_model(round_x), round_y)
round_opt.zero_grad(); round_loss.backward(); round_opt.step()
round_model.eval()
with torch.inference_mode():
    original_output = round_model(round_x)

round_path = os.path.join(tmpdir, "round_trip_weights.pt")
torch.save(round_model.state_dict(), round_path)
round_restored = SimpleClassifier()
round_restored.load_state_dict(torch.load(round_path, map_location="cpu", weights_only=True))
round_restored.eval()
with torch.inference_mode():
    restored_output = round_restored(round_x)
assert torch.allclose(original_output, restored_output)
print("✅ Save → new model → load → identical output")


## 9. Saving Full Training Checkpoints 🟡

| Weights-only | Training checkpoint |
|---|---|
| Model state | Model state plus training state |
| Small and simple | Larger and richer |
| Best for inference | Best for exact resume |
| Architecture still required | Architecture still generally required |

A checkpoint can include model, optimizer, epoch/global step, validation metric, scheduler, AMP scaler, configuration, preprocessing, and label metadata.


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
# Populate AdamW's internal running statistics with one real update.
train_x, train_y = torch.randn(32, 10), torch.randint(0, 3, (32,))
optimizer.zero_grad()
loss = nn.CrossEntropyLoss()(model(train_x), train_y)
loss.backward(); optimizer.step()

checkpoint_path = os.path.join(tmpdir, "checkpoint.pt")
checkpoint_to_save = {
    "epoch": 5,
    "global_step": 192,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "train_loss": float(loss.item()),
    "val_loss": 0.31,
}
torch.save(checkpoint_to_save, checkpoint_path)
print("Saved keys:", checkpoint_to_save.keys())


## 10. Optimizer State 🔥 Interview Essential


In [ ]:
opt_state = optimizer.state_dict()
print("Parameter groups:", len(opt_state["param_groups"]))
print("Per-parameter state entries:", len(opt_state["state"]))
assert len(opt_state["state"]) > 0


Adam/AdamW and SGD with momentum remember optimization history. Loading only model weights preserves parameters but resets that history, so the continuation is not exact. Plain SGD without momentum may have little optimizer state.

General rule: checkpoint every stateful object needed to reproduce the next update. If used, that may include `scheduler.state_dict()` and CUDA FP16 `scaler.state_dict()`.


## 11. Resume Training 🔥 Interview Essential


In [ ]:
# This file was created locally above and is trusted. Never use weights_only=False
# on an untrusted checkpoint: Python serialization can have security implications.
restored_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
resumed_model = SimpleClassifier().to(device)
resumed_optimizer = optim.AdamW(resumed_model.parameters(), lr=1e-3)
resumed_model.load_state_dict(restored_checkpoint["model_state_dict"])
resumed_optimizer.load_state_dict(restored_checkpoint["optimizer_state_dict"])
start_epoch = restored_checkpoint["epoch"] + 1
resumed_model.train()
print("Resume at epoch", start_epoch)


### Exercise 3 — Train 3 epochs, save, then resume for 2


In [ ]:
torch.manual_seed(123)
resume_X = torch.randn(96, 10).to(device)
resume_y = torch.randint(0, 3, (96,)).to(device)

def train_epochs(model, optimizer, X, y, start, stop):
    losses = []
    model.train()
    for epoch in range(start, stop):
        optimizer.zero_grad()
        batch_loss = nn.CrossEntropyLoss()(model(X), y)
        batch_loss.backward(); optimizer.step()
        losses.append(float(batch_loss.item()))
    return losses

phase1_model = SimpleClassifier().to(device)
phase1_optimizer = optim.AdamW(phase1_model.parameters(), lr=1e-2)
first_losses = train_epochs(phase1_model, phase1_optimizer, resume_X, resume_y, 0, 3)
saved_epoch = 2
resume_path = os.path.join(tmpdir, "resume_exercise.pt")
torch.save({
    "epoch": saved_epoch,
    "model_state_dict": phase1_model.state_dict(),
    "optimizer_state_dict": phase1_optimizer.state_dict(),
}, resume_path)

restored_model = SimpleClassifier().to(device)
restored_optimizer = optim.AdamW(restored_model.parameters(), lr=1e-2)
resume_ckpt = torch.load(resume_path, map_location=device, weights_only=False)
restored_model.load_state_dict(resume_ckpt["model_state_dict"])
restored_optimizer.load_state_dict(resume_ckpt["optimizer_state_dict"])
start_epoch = resume_ckpt["epoch"] + 1
second_losses = train_epochs(restored_model, restored_optimizer, resume_X, resume_y, start_epoch, start_epoch + 2)

assert start_epoch == saved_epoch + 1
assert len(restored_optimizer.state_dict()["state"]) > 0
assert len(second_losses) == 2
print("✅ Continued epochs", list(range(start_epoch, start_epoch + 2)), "losses", second_losses)


### Exercise 4 — Full checkpoint round trip


In [ ]:
round_config = {"input_dim": 10, "hidden_dim": 32, "num_classes": 3}
original_epoch = 4
checkpoint_round_path = os.path.join(tmpdir, "checkpoint_round_trip.pt")
torch.save({
    "model_state_dict": restored_model.state_dict(),
    "optimizer_state_dict": restored_optimizer.state_dict(),
    "epoch": original_epoch,
    "loss": second_losses[-1],
    "config": round_config,
}, checkpoint_round_path)

round_checkpoint = torch.load(checkpoint_round_path, map_location=device, weights_only=False)
round_checkpoint_model = SimpleClassifier(**round_checkpoint["config"]).to(device)
round_checkpoint_optimizer = optim.AdamW(round_checkpoint_model.parameters(), lr=1e-2)
round_checkpoint_model.load_state_dict(round_checkpoint["model_state_dict"])
round_checkpoint_optimizer.load_state_dict(round_checkpoint["optimizer_state_dict"])
restored_epoch = round_checkpoint["epoch"]
restored_config = round_checkpoint["config"]

restored_model.eval(); round_checkpoint_model.eval()
with torch.inference_mode():
    source_logits = restored_model(resume_X[:8])
    checkpoint_logits = round_checkpoint_model(resume_X[:8])
assert restored_epoch == original_epoch
assert restored_config == round_config
assert round_checkpoint["loss"] == second_losses[-1]
assert torch.allclose(source_logits, checkpoint_logits)
print("✅ Model, optimizer, epoch, loss, and config restored")


## 12. Saving the Best Model 🔥 Interview Essential


In [ ]:
best_model_path = os.path.join(tmpdir, "model_best.pt")
last_model_path = os.path.join(tmpdir, "checkpoint_last.pt")
best_val_loss = float("inf")
simulated_val_losses = [0.62, 0.48, 0.51, 0.44]
for epoch, val_loss in enumerate(simulated_val_losses):
    if val_loss < best_val_loss:  # loss: lower is better
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
    torch.save({"epoch": epoch, "model_state_dict": model.state_dict()}, last_model_path)

assert best_val_loss == min(simulated_val_losses)
print("Best validation loss:", best_val_loss)


The final epoch is not necessarily the best. Save **best** for evaluation/deployment candidacy and **last** for the exact most recent training state. For accuracy, reverse the comparison: save when `val_accuracy > best_val_accuracy`.


## 13. `map_location` 🔥 Interview Essential


In [ ]:
portable_state = torch.load(weights_path, map_location="cpu", weights_only=True)
portable_model = SimpleClassifier()
portable_model.load_state_dict(portable_state)
portable_model.to("cpu").eval()
print(next(portable_model.parameters()).device)


`map_location` controls where tensors are reconstructed. It is the standard fix for loading a CUDA-created checkpoint on a CPU-only host.


## 14. Loading GPU Checkpoints on CPU


```python
state = torch.load(path, map_location="cpu", weights_only=True)
model = Model(...)
model.load_state_dict(state)
model.to("cpu")
```

For CPU → GPU/MPS, load with `map_location=device`, copy the state, and ensure the model and inputs are on the same device. Loading then moving is also valid.


## 15. `strict=True` vs `strict=False` 🟡


In [ ]:
class ExtendedClassifier(SimpleClassifier):
    def __init__(self):
        super().__init__()
        self.extra_scale = nn.Parameter(torch.ones(1))

extended = ExtendedClassifier()
non_strict_result = extended.load_state_dict(model.state_dict(), strict=False)
print("Missing:", non_strict_result.missing_keys)
print("Unexpected:", non_strict_result.unexpected_keys)
assert non_strict_result.missing_keys == ["extra_scale"]


`strict=True` (default) requires exact expected keys and no unexpected keys. `strict=False` permits missing/unexpected keys—useful for transfer learning—but **does not automatically resolve a shape mismatch for a matching key**.


## 16. Missing and Unexpected Keys


In [ ]:
state_with_extra = dict(model.state_dict())
state_with_extra["retired_head.weight"] = torch.randn(2, 2)
result = SimpleClassifier().load_state_dict(state_with_extra, strict=False)
print("Missing:", result.missing_keys)
print("Unexpected:", result.unexpected_keys)


Checkpoint lacks a current parameter → **missing key**. Checkpoint has a parameter absent from the current model → **unexpected key**. Prefix differences such as `module.`, `model.`, or `backbone.` can create both.


## 17. Architecture Mismatch and Transfer Learning 🔴 Optional


In [ ]:
class BackboneModel(nn.Module):
    def __init__(self, input_dim=10, hidden_size=16, num_classes=5):
        super().__init__()
        self.backbone = nn.Sequential(nn.Linear(input_dim, hidden_size), nn.ReLU())
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        return self.classifier(self.backbone(x))

old_model = BackboneModel(num_classes=5)
old_state = old_model.state_dict()
new_head_model = BackboneModel(num_classes=3)

# Keep only keys whose names and shapes are compatible.
new_state = new_head_model.state_dict()
compatible = {k: v for k, v in old_state.items() if k in new_state and v.shape == new_state[k].shape}
result = new_head_model.load_state_dict(compatible, strict=False)

assert torch.equal(new_head_model.backbone[0].weight, old_model.backbone[0].weight)
assert new_head_model.classifier.weight.shape == (3, 16)
assert set(result.missing_keys) == {"classifier.weight", "classifier.bias"}
print("✅ Backbone restored; new 3-class head retained")


A 5-class head cannot populate a 3-class head because their tensors differ. Filter compatible keys or load the backbone submodule, then initialize/train the new head.


## 18. Inference Mode 🔥 Interview Essential


In [ ]:
inference_model = SimpleClassifier().to(device)
inference_model.eval()  # changes Dropout/BatchNorm behavior
inference_X = torch.randn(6, 10, device=device)
with torch.inference_mode():  # disables Autograd overhead
    inference_logits = inference_model(inference_X)
assert not inference_logits.requires_grad
print(inference_logits.shape)


Use both. `eval()` does **not** disable gradient tracking; `inference_mode()` does **not** switch Dropout/BatchNorm behavior. `torch.no_grad()` also disables gradient recording, while `inference_mode()` enables additional inference optimizations and stricter tensor semantics.


## 19. Single-Sample Inference 🟢


In [ ]:
x_one = torch.randn(10)
x_batch = x_one.unsqueeze(0).to(device)
inference_model.eval()
with torch.inference_mode():
    one_logits = inference_model(x_batch)
assert one_logits.shape == (1, 3)
print("single → batch:", tuple(x_one.shape), "→", tuple(x_batch.shape))


## 20. Batch and Multiclass Inference


In [ ]:
with torch.inference_mode():
    logits = inference_model(inference_X)
    probabilities = torch.softmax(logits, dim=1)
    predictions = logits.argmax(dim=1)
    top_probs, top_classes = torch.topk(probabilities, k=3, dim=1)
assert predictions.shape == (6,)
assert torch.allclose(probabilities.sum(1), torch.ones(6, device=device), atol=1e-6)
print("predictions:", predictions.cpu().tolist())


## 21. Binary Classification Inference


In [ ]:
binary_model = nn.Linear(10, 1).to(device)
binary_model.eval()
with torch.inference_mode():
    binary_logits = binary_model(inference_X)
    binary_probs = torch.sigmoid(binary_logits)
    binary_pred = (binary_probs >= 0.5).long()
assert binary_probs.min() >= 0 and binary_probs.max() <= 1
assert set(binary_pred.cpu().flatten().tolist()) <= {0, 1}


Training: raw logits → `BCEWithLogitsLoss`. Inference: logits → sigmoid → chosen threshold.


## 22. Multiclass Inference


Use `argmax(dim=1)` for class indices and `softmax(dim=1)` only when probabilities are needed. Cross-entropy consumes raw logits; applying softmax before it is unnecessary and less numerically stable.


## 23. Regression Inference


In [ ]:
regression_model = nn.Linear(10, 1).to(device).eval()
with torch.inference_mode():
    regression_predictions = regression_model(inference_X).cpu()
assert regression_predictions.shape == (6, 1)


Regression needs no sigmoid/softmax unless the target definition specifically requires one. Apply domain-specific inverse scaling after prediction when training targets were scaled.


## 24. Text-Model Inference (Conceptual)


The same lifecycle applies to text models: preserve tokenizer/vocabulary, padding/truncation policy, special-token IDs, model config, and label mapping. This notebook stays offline and does not introduce a model-serving or Hugging Face serialization workflow.


## 25. Preprocessing Consistency 🔥 Interview Essential


In [ ]:
raw_train = torch.randn(100, 10) * 3 + 2
train_mean = raw_train.mean(dim=0)
train_std = raw_train.std(dim=0).clamp_min(1e-6)
new_raw = torch.randn(4, 10) * 3 + 2
new_normalized = (new_raw - train_mean) / train_std

metadata_path = os.path.join(tmpdir, "inference_artifact.pt")
config = {"input_dim": 10, "hidden_dim": 32, "num_classes": 3}
class_names = ["cat", "dog", "bird"]
torch.save({
    "model_state_dict": model.state_dict(),
    "feature_mean": train_mean,
    "feature_std": train_std,
    "config": config,
    "class_names": class_names,
}, metadata_path)
print("Saved preprocessing, config, and label semantics")


Never recompute normalization from one inference sample. Preserve training mean/std, feature order/schema, tokenizer or transforms, and index-to-label mapping. A numeric class is useless if its label meaning is lost.


## 26. Device-Aware Inference


In [ ]:
def predict_multiclass(model, X, device):
    model = model.to(device)
    model.eval()
    with torch.inference_mode():
        logits = model(X.to(device))
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)
    return pred.cpu(), probs.cpu()

pred, prob = predict_multiclass(SimpleClassifier(), torch.randn(5, 10), device)
assert pred.shape == (5,) and prob.shape == (5, 3)
assert torch.allclose(prob.sum(1), torch.ones(5), atol=1e-6)
print("✅ Device-aware API validated")


In [ ]:
def predict_binary(model, X, device, threshold=0.5):
    model = model.to(device).eval()
    with torch.inference_mode():
        probs = torch.sigmoid(model(X.to(device)))
        preds = (probs >= threshold).long()
    return probs.cpu(), preds.cpu()

def predict_regression(model, X, device):
    model = model.to(device).eval()
    with torch.inference_mode():
        return model(X.to(device)).cpu()

bp, by = predict_binary(nn.Linear(10, 1), torch.randn(7, 10), device)
rp = predict_regression(nn.Linear(10, 1), torch.randn(7, 10), device)
assert bp.min() >= 0 and bp.max() <= 1 and set(by.flatten().tolist()) <= {0, 1}
assert rp.shape == (7, 1)


## 27. Efficient Large-Batch Inference


In [ ]:
batch_data = TensorDataset(torch.randn(513, 10))
inference_loader = DataLoader(batch_data, batch_size=128, shuffle=False)
batch_model = SimpleClassifier().to(device).eval()
all_predictions, all_probabilities = [], []
with torch.inference_mode():
    for (X_batch,) in inference_loader:
        logits = batch_model(X_batch.to(device))
        all_predictions.append(logits.argmax(1).cpu())
        all_probabilities.append(torch.softmax(logits, 1).cpu())
all_predictions = torch.cat(all_predictions)
all_probabilities = torch.cat(all_probabilities)
assert all_predictions.shape == (513,) and all_probabilities.shape == (513, 3)


Move results to CPU as you collect them; retaining thousands of GPU tensors exhausts accelerator memory. `shuffle=False` preserves input/prediction alignment. Larger batches may raise throughput but use more memory and can increase request latency. Benchmark the workload—largest is not automatically best.


In [ ]:
# Lightweight warm-up and timing pattern. Synchronize CUDA around real measurements.
timing_X = torch.randn(128, 10, device=device)
with torch.inference_mode():
    for _ in range(3):
        _ = batch_model(timing_X)
if device.type == "cuda": torch.cuda.synchronize()
start = time.perf_counter()
with torch.inference_mode():
    for _ in range(10):
        _ = batch_model(timing_X)
if device.type == "cuda": torch.cuda.synchronize()
print(f"10 batches: {time.perf_counter() - start:.4f}s")


## 28. Saving Predictions


In [ ]:
predictions_path = os.path.join(tmpdir, "predictions.pt")
torch.save({"predictions": all_predictions, "probabilities": all_probabilities}, predictions_path)
saved_predictions = torch.load(predictions_path, map_location="cpu", weights_only=True)
assert saved_predictions["predictions"].device.type == "cpu"
print("Saved", len(saved_predictions["predictions"]), "ordered predictions")


## 29. Checkpoint Management


Prefer structured names: `model_best.pt`, `checkpoint_last.pt`, `checkpoint_epoch_010.pt`, `checkpoint_step_050000.pt`—not `final_final_v2_REAL.pt`.

- **Best:** best chosen validation metric; likely evaluation/deployment candidate.
- **Last:** most recent state; useful for interruption recovery.
- Save frequency trades I/O/storage against recoverable work.
- Production systems often write a temporary checkpoint, verify it, then atomically rename it.
- Track code/data versions, hyperparameters, feature schema, tokenizer, configuration, and metrics where relevant.

PyTorch can serialize whole model objects, but state-dict workflows are clearer, more flexible, and less tied to Python class serialization. Never blindly load untrusted checkpoint files; prefer trusted artifacts and current safer loading options.


## 30. Common Mistakes


1. Forgetting `eval()` leaves Dropout/BatchNorm in training behavior.
2. Forgetting `inference_mode()` records needless Autograd state.
3. Rebuilding the wrong hidden size or class count causes shape errors.
4. Saving only weights cannot exactly restore AdamW/scheduler/scaler progress.
5. Omitting `map_location` can fail on a machine without the saved device.
6. Recomputing preprocessing or changing class mappings silently corrupts semantics.
7. A single `(features,)` input may need a `(1, features)` batch dimension.
8. Keeping outputs on GPU consumes limited accelerator memory.
9. Freezing loaded parameters (`requires_grad=False`) prevents fine-tuning until explicitly unfrozen.


## 31. Debugging Exercises (27 scenarios)


### Debug 1 — Predictions change across calls

**Broken code**

```python
loaded_model(X)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Dropout may still be active.

**Fix:** Call `loaded_model.eval()` before inference.

</details>


### Debug 2 — Inference uses excess memory

**Broken code**

```python
for X in loader: outputs.append(model(X))
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Autograd graphs are recorded.

**Fix:** Wrap the loop in `torch.inference_mode()` and store CPU outputs.

</details>


### Debug 3 — There is nowhere to load weights

**Broken code**

```python
model = torch.load(weights_path)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** A weights-only file is not an `nn.Module`.

**Fix:** Construct the architecture, load the state dict, then call `load_state_dict`.

</details>


### Debug 4 — Size mismatch in first/last linear layers

**Broken code**

```python
SimpleClassifier(hidden_dim=64).load_state_dict(state32)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The reconstructed hidden size differs.

**Fix:** Rebuild with the saved config (`hidden_dim=32`).

</details>


### Debug 5 — Classifier size mismatch

**Broken code**

```python
Model(num_classes=5).load_state_dict(state10)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The class count changed.

**Fix:** Use the original class count or intentionally replace/reinitialize the head.

</details>


### Debug 6 — Resume behaves differently

**Broken code**

```python
torch.save(model.state_dict(), path)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Optimizer history was never saved.

**Fix:** Save and restore `optimizer.state_dict()` plus epoch/step.

</details>


### Debug 7 — CUDA deserialization fails on CPU

**Broken code**

```python
torch.load(path)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Saved tensors reference an unavailable CUDA device.

**Fix:** Use `map_location='cpu'`.

</details>


### Debug 8 — KeyError: optimizer_state_dict

**Broken code**

```python
checkpoint['optimizer_state_dict']
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The file is weights-only or uses a different schema.

**Fix:** Inspect `checkpoint.keys()` and load according to its schema.

</details>


### Debug 9 — Unexpected key `retired_head.weight`

**Broken code**

```python
model.load_state_dict(state)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Checkpoint contains a removed module.

**Fix:** Confirm architecture/version; use intentional partial loading if appropriate.

</details>


### Debug 10 — Strict loading rejects a new head

**Broken code**

```python
model.load_state_dict(backbone_state, strict=True)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The current model expects keys absent from the checkpoint.

**Fix:** Use `strict=False`, then inspect missing/unexpected keys.

</details>


### Debug 11 — `strict=False` still raises size mismatch

**Broken code**

```python
new5.load_state_dict(old4, strict=False)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Matching head keys have incompatible shapes.

**Fix:** Filter head keys or load only the backbone.

</details>


### Debug 12 — Index 2 displays the wrong label

**Broken code**

```python
label = new_names[pred]
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The deployment label mapping differs from training.

**Fix:** Restore the checkpoint's exact class mapping.

</details>


### Debug 13 — Accuracy collapses after deployment

**Broken code**

```python
x = (x - x.mean()) / x.std()
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Per-sample statistics differ from training preprocessing.

**Fix:** Use saved training statistics and feature order.

</details>


### Debug 14 — Expected all tensors on same device

**Broken code**

```python
model.to(device); model(X_cpu)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Input and parameters are on different devices.

**Fix:** Move the batch with `X.to(device)`.

</details>


### Debug 15 — Linear layer sees 1-D input unexpectedly downstream

**Broken code**

```python
logits = model(torch.randn(10))
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** A batch dimension is missing.

**Fix:** Use `x.unsqueeze(0)` to obtain `(1, 10)`.

</details>


### Debug 16 — Binary scores are outside [0, 1]

**Broken code**

```python
probs = model(X)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Raw logits were presented as probabilities.

**Fix:** Apply `torch.sigmoid(logits)` then threshold.

</details>


### Debug 17 — Multiclass probabilities do not compete

**Broken code**

```python
probs = torch.sigmoid(logits)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Sigmoid treats classes independently.

**Fix:** For mutually exclusive classes, use `softmax(logits, dim=1)`.

</details>


### Debug 18 — Each column sums to 1

**Broken code**

```python
softmax(logits, dim=0)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Softmax used the batch dimension.

**Fix:** Use `dim=1` for logits shaped `(B, C)`.

</details>


### Debug 19 — Predictions no longer align with record IDs

**Broken code**

```python
DataLoader(ds, shuffle=True)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Inference order was shuffled.

**Fix:** Use `shuffle=False` or carry stable IDs.

</details>


### Debug 20 — GPU memory grows every batch

**Broken code**

```python
all_logits.append(logits)
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Accelerator tensors are retained.

**Fix:** Append `logits.cpu()` inside inference mode.

</details>


### Debug 21 — Earlier checkpoint disappears

**Broken code**

```python
torch.save(ckpt, 'checkpoint.pt')
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** A fixed filename was overwritten.

**Fix:** Use best/last semantics or epoch/step/versioned filenames.

</details>


### Debug 22 — Worst loss is saved as best

**Broken code**

```python
if val_loss > best_val_loss: save()
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** The comparison is reversed for loss.

**Fix:** Save when `val_loss < best_val_loss`.

</details>


### Debug 23 — No checkpoint is ever saved

**Broken code**

```python
best_val_loss = 0; if val_loss < best_val_loss: save()
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Loss cannot beat the incorrect initial sentinel.

**Fix:** Initialize `best_val_loss = float('inf')`.

</details>


### Debug 24 — Epoch 5 trains twice

**Broken code**

```python
start_epoch = checkpoint['epoch']
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Saved epoch is the completed epoch.

**Fix:** Use `start_epoch = saved_epoch + 1`.

</details>


### Debug 25 — Optimizer state does not map to new parameters

**Broken code**

```python
optimizer = AdamW(old_model.parameters()); model = NewModel()
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Optimizer was attached to a discarded model.

**Fix:** Construct the new model, construct its optimizer, then load both state dicts.

</details>


### Debug 26 — Optimizer state dictionary is empty

**Broken code**

```python
optimizer = AdamW(model.parameters()); save()
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** No optimizer step has populated moment state yet.

**Fix:** This is expected before the first relevant step; save after updates for a meaningful resume test.

</details>


### Debug 27 — Loaded model never updates

**Broken code**

```python
for p in model.parameters(): p.requires_grad = False
```

**Question:** What is the likely cause, and how would you verify it?

<details><summary>Hidden answer and fix</summary>

**Answer:** Parameters were intentionally or accidentally frozen.

**Fix:** Unfreeze intended parameters before building/using the optimizer.

</details>


## 32. Interview Questions (42)


### 1. What is a PyTorch `state_dict`?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** A mapping from parameter/buffer names to tensors.

**Detailed Explanation:** It represents module state but not the Python architecture definition.

</details>


### 2. What does `model.state_dict()` contain?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Parameters and persistent buffers.

**Detailed Explanation:** It includes learned weights/biases and buffers such as BatchNorm running statistics.

</details>


### 3. Does it contain model architecture?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** No.

**Detailed Explanation:** Recreate compatible code/config before loading.

</details>


### 4. How do you save model weights?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** `torch.save(model.state_dict(), path)`.

**Detailed Explanation:** This produces a portable weights-only artifact.

</details>


### 5. How do you load model weights?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Load the dict, then call `model.load_state_dict(state)`.

**Detailed Explanation:** Use `map_location` and current safer loading options.

</details>


### 6. Why must architecture exist first?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** The dict only names and values state.

**Detailed Explanation:** The module owns the computation graph and destination tensors.

</details>


### 7. Why prefer state-dict saving?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** It is explicit and flexible.

**Detailed Explanation:** It is less coupled to pickled Python class definitions than whole-object serialization.

</details>


### 8. What belongs in a training checkpoint?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Model, optimizer, epoch/step, and other state.

**Detailed Explanation:** Often scheduler/scaler, metrics, config, preprocessing, and version metadata too.

</details>


### 9. Why save optimizer state?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** To retain update history.

**Detailed Explanation:** Momentum and Adam moments affect future parameter updates.

</details>


### 10. What if Adam state is not restored?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Training can continue but not as the exact run.

**Detailed Explanation:** Moment estimates restart even though weights match.

</details>


### 11. How do you resume training?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Rebuild model/optimizer, load both states, restore counters, call `train()`.

**Detailed Explanation:** Also restore every stateful scheduler/scaler/randomness requirement used by the run.

</details>


### 12. Why save epoch/global step?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** To continue counters and schedules correctly.

**Detailed Explanation:** It also supports logging, naming, and reproducibility.

</details>


### 13. Best vs last checkpoint?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Best optimizes a validation metric; last is most recent.

**Detailed Explanation:** They serve deployment selection and interruption recovery respectively.

</details>


### 14. How do you save the best model?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Compare each validation metric against the best so far.

**Detailed Explanation:** Save on lower loss or higher score, depending on metric direction.

</details>


### 15. What is `map_location`?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** A load-time tensor device mapping.

**Detailed Explanation:** It enables checkpoint portability across CUDA, MPS, and CPU.

</details>


### 16. Load a GPU checkpoint on CPU?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Use `map_location='cpu'`.

**Detailed Explanation:** Then load into a CPU model and pass CPU inputs.

</details>


### 17. What does `strict=True` mean?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Expected and supplied keys must match exactly.

**Detailed Explanation:** It is safest when the architecture should be identical.

</details>


### 18. When is `strict=False` useful?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Intentional partial loading.

**Detailed Explanation:** Examples include a reused backbone or newly replaced classifier head.

</details>


### 19. What are missing keys?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Current model keys absent from the checkpoint.

**Detailed Explanation:** They remain initialized unless populated another way.

</details>


### 20. What are unexpected keys?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Checkpoint keys absent from the current model.

**Detailed Explanation:** They often indicate version or prefix differences.

</details>


### 21. Can `strict=False` fix shape mismatches?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** No, not for matching names with incompatible tensors.

**Detailed Explanation:** Remove/filter those keys or load only compatible submodules.

</details>


### 22. How do you replace a pretrained head?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Load compatible backbone weights and initialize a new head.

**Detailed Explanation:** Then train the head and optionally fine-tune the backbone.

</details>


### 23. What does `model.eval()` do?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Switches modules to evaluation behavior.

**Detailed Explanation:** It affects modules such as Dropout and BatchNorm; it does not disable Autograd.

</details>


### 24. What does `inference_mode()` do?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Disables gradient tracking with inference optimizations.

**Detailed Explanation:** It does not switch module training/evaluation behavior.

</details>


### 25. Why use both?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** They solve separate correctness and efficiency problems.

**Detailed Explanation:** Use `eval()` plus an `inference_mode()` context.

</details>


### 26. Binary inference?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Sigmoid logits, then threshold.

**Detailed Explanation:** Match output shape and threshold policy to the problem.

</details>


### 27. Multiclass inference?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Argmax logits; softmax when probabilities are needed.

**Detailed Explanation:** For `(B,C)`, class operations use `dim=1`.

</details>


### 28. Why train on logits but present probabilities?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Losses use stable fused logit math; users may need interpretable scores.

**Detailed Explanation:** Do not add softmax before CrossEntropyLoss.

</details>


### 29. How do you handle one sample?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Add a batch dimension.

**Detailed Explanation:** For `(10,)`, use `unsqueeze(0)` to get `(1,10)`.

</details>


### 30. Why must preprocessing match?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** The model learned under those input assumptions.

**Detailed Explanation:** Changed scaling, tokenization, or feature order creates distribution/semantic errors.

</details>


### 31. Why save class mappings?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Indices have no inherent label meaning.

**Detailed Explanation:** A changed mapping can make correct math produce wrong application output.

</details>


### 32. Why save model config?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** To reconstruct the compatible architecture.

**Detailed Explanation:** It reduces hidden-size/class-count/version mistakes.

</details>


### 33. What makes a reproducible checkpoint?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** All state and metadata needed to recreate the next action.

**Detailed Explanation:** Include model/training state plus code/data/config/preprocessing identifiers as appropriate.

</details>


### 34. Why `shuffle=False` for inference?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** To preserve input-output order.

**Detailed Explanation:** Alternatively propagate stable IDs through the pipeline.

</details>


### 35. What is latency?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Time to produce a response.

**Detailed Explanation:** Usually measured per request or percentile distribution.

</details>


### 36. What is throughput?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Samples/requests processed per unit time.

**Detailed Explanation:** Batching often improves it.

</details>


### 37. How does batching affect both?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** It often raises throughput but may add waiting/latency.

**Detailed Explanation:** Benchmark under realistic arrival patterns and memory constraints.

</details>


### 38. Why move outputs to CPU?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** To free scarce accelerator memory and support downstream storage.

**Detailed Explanation:** Do so batch by batch during inference.

</details>


### 39. How do you diagnose load failure?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Inspect config, checkpoint/state keys, shapes, prefixes, and devices.

**Detailed Explanation:** Use strict-load errors and non-strict missing/unexpected reports deliberately.

</details>


### 40. Security concern with checkpoint files?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Python-based deserialization can be unsafe for untrusted files.

**Detailed Explanation:** Use trusted sources and safer current loading modes where possible.

</details>


### 41. How would you version checkpoints?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** Use structured best/last/epoch/step names plus metadata.

**Detailed Explanation:** Track model, code, data, and schema versions—not only filenames.

</details>


### 42. Why retain best and last?
<details><summary>Short interview answer + detailed explanation</summary>

**Short Interview Answer:** They optimize different recovery/deployment goals.

**Detailed Explanation:** Best may be earlier; last is needed to resume the latest work.

</details>


## 33. Knowledge Check Quiz (30)


**Q1. What does `torch.save(model.state_dict(), path)` normally save?**  
A. Model source code  
B. Parameter/buffer state  
C. Dataset  
D. Interpreter


**Q2. Which helps load CUDA tensors on CPU?**  
A. shuffle=False  
B. map_location='cpu'  
C. model.train()  
D. requires_grad=False


**Q3. What does `model.eval()` NOT do?**  
A. Alter Dropout  
B. Alter BatchNorm  
C. Disable Autograd  
D. Set evaluation behavior


**Q4. Why save optimizer state?**  
A. Labels  
B. Internal optimizer history/config  
C. Architecture  
D. Inference mode


**Q5. Which reconstructs the architecture?**  
A. state_dict alone  
B. Python model class/config  
C. optimizer state  
D. softmax


**Q6. Default strict loading requires what?**  
A. Only shapes  
B. Exact compatible keys  
C. CUDA  
D. An optimizer


**Q7. A missing key means what?**  
A. Checkpoint lacks a model-expected key  
B. File missing  
C. GPU missing  
D. Label missing


**Q8. An unexpected key means what?**  
A. Current model has extra key  
B. Checkpoint has an unknown key  
C. Tensor is NaN  
D. Epoch is wrong


**Q9. Does `strict=False` ignore matching-name shape mismatch?**  
A. Always  
B. Never loads anything  
C. No  
D. Only on CPU


**Q10. Correct multiclass probability transform?**  
A. sigmoid  
B. softmax over classes  
C. argmin  
D. round


**Q11. Correct binary probability transform?**  
A. softmax dim=0  
B. sigmoid  
C. argmax  
D. none ever


**Q12. Which mode changes Dropout?**  
A. inference_mode only  
B. eval  
C. no_grad only  
D. optimizer.zero_grad


**Q13. Which disables gradient recording?**  
A. eval  
B. train  
C. inference_mode  
D. state_dict


**Q14. Why use `shuffle=False` in ordered inference?**  
A. Accuracy  
B. Preserve correspondence  
C. Enable CUDA  
D. Save memory


**Q15. Best checkpoint under validation loss uses what comparison?**  
A. >  
B. <  
C. == only  
D. random


**Q16. Best checkpoint under accuracy uses what comparison?**  
A. >  
B. <  
C. !=  
D. none


**Q17. A completed saved epoch 7 resumes at what epoch?**  
A. 6  
B. 7  
C. 8  
D. 0


**Q18. What should a one-feature-vector batch add?**  
A. Class dimension  
B. Batch dimension  
C. Device string  
D. Optimizer


**Q19. Why preserve training mean/std?**  
A. File compression  
B. Preprocessing consistency  
C. Faster save  
D. More classes


**Q20. Why preserve class names?**  
A. Optimizer needs them  
B. Map indices to semantics  
C. Autograd needs them  
D. CUDA requires them


**Q21. What is usually best for resuming?**  
A. Weights only  
B. Full training checkpoint  
C. Predictions only  
D. Class names only


**Q22. What is usually simplest for inference?**  
A. Weights plus architecture/config  
B. Optimizer only  
C. Epoch only  
D. Scheduler only


**Q23. Where should accumulated accelerator outputs go?**  
A. Stay on GPU forever  
B. CPU  
C. Optimizer state  
D. Gradients


**Q24. Latency measures what?**  
A. Samples per second  
B. Time per response  
C. Checkpoint size  
D. Epoch count


**Q25. Throughput measures what?**  
A. Responses/samples per time  
B. Single request time  
C. Loss  
D. Memory only


**Q26. Can final epoch differ from best epoch?**  
A. No  
B. Yes  
C. Only on CUDA  
D. Only regression


**Q27. What might `module.` keys indicate?**  
A. A wrapper/prefix difference  
B. Corruption always  
C. Softmax  
D. Normalization


**Q28. What does whole-model serialization couple to more strongly?**  
A. Batch size  
B. Python class/import details  
C. Loss scale  
D. Labels only


**Q29. Should untrusted checkpoint files be blindly loaded?**  
A. Yes  
B. Only on GPU  
C. No  
D. Only if small


**Q30. What is TorchScript/`torch.compile` here?**  
A. Required saving format  
B. High-level deployment/performance awareness  
C. Optimizer  
D. Dataset


### Quiz answers

1:B · 2:B · 3:C · 4:B · 5:B · 6:B · 7:A · 8:B · 9:C · 10:B · 11:B · 12:B · 13:C · 14:B · 15:B · 16:A · 17:C · 18:B · 19:B · 20:B · 21:B · 22:A · 23:B · 24:B · 25:A · 26:B · 27:A · 28:B · 29:C · 30:B


## 34. Write From Memory (25 prompts)


1. **Save a model state dict.**  
Write the code without scrolling up, then compare against the cheat sheet.


2. **Load a state dict on CPU safely.**  
Write the code without scrolling up, then compare against the cheat sheet.


3. **Recreate a model and load weights.**  
Write the code without scrolling up, then compare against the cheat sheet.


4. **Put a model in evaluation mode.**  
Write the code without scrolling up, then compare against the cheat sheet.


5. **Run inference without Autograd.**  
Write the code without scrolling up, then compare against the cheat sheet.


6. **Save model + optimizer + epoch.**  
Write the code without scrolling up, then compare against the cheat sheet.


7. **Restore a full checkpoint.**  
Write the code without scrolling up, then compare against the cheat sheet.


8. **Restore AdamW state.**  
Write the code without scrolling up, then compare against the cheat sheet.


9. **Calculate `start_epoch`.**  
Write the code without scrolling up, then compare against the cheat sheet.


10. **Save only when validation loss improves.**  
Write the code without scrolling up, then compare against the cheat sheet.


11. **Load with `strict=False`.**  
Write the code without scrolling up, then compare against the cheat sheet.


12. **Print missing/unexpected keys.**  
Write the code without scrolling up, then compare against the cheat sheet.


13. **Convert `(10,)` to `(1,10)`.**  
Write the code without scrolling up, then compare against the cheat sheet.


14. **Convert multiclass logits to predictions.**  
Write the code without scrolling up, then compare against the cheat sheet.


15. **Convert multiclass logits to probabilities.**  
Write the code without scrolling up, then compare against the cheat sheet.


16. **Convert binary logits to probabilities and labels.**  
Write the code without scrolling up, then compare against the cheat sheet.


17. **Move inference output to CPU.**  
Write the code without scrolling up, then compare against the cheat sheet.


18. **Build ordered batch inference.**  
Write the code without scrolling up, then compare against the cheat sheet.


19. **Preserve inference record order.**  
Write the code without scrolling up, then compare against the cheat sheet.


20. **Save train mean/std.**  
Write the code without scrolling up, then compare against the cheat sheet.


21. **Save class names.**  
Write the code without scrolling up, then compare against the cheat sheet.


22. **Save model configuration.**  
Write the code without scrolling up, then compare against the cheat sheet.


23. **Filter compatible backbone keys.**  
Write the code without scrolling up, then compare against the cheat sheet.


24. **Keep best and last checkpoint names.**  
Write the code without scrolling up, then compare against the cheat sheet.


25. **Verify restored outputs match.**  
Write the code without scrolling up, then compare against the cheat sheet.


## 35. Final Checkpoint + Inference Challenge 🔴


This end-to-end task trains a synthetic 4-class model, saves best and last checkpoints, simulates a new program, verifies restoration, resumes training, and performs ordered batch inference.


In [ ]:
torch.manual_seed(42)
X = torch.randn(1200, 20)
true_weights = torch.randn(20, 4)
y = (X @ true_weights).argmax(dim=1)
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_val, y_val = X[split:], y[split:]

final_config = {"input_dim": 20, "hidden1": 64, "hidden2": 32, "num_classes": 4}
final_class_names = ["class_A", "class_B", "class_C", "class_D"]

class Classifier(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(),
            nn.Linear(hidden1, hidden2), nn.ReLU(),
            nn.Linear(hidden2, num_classes),
        )
    def forward(self, x):
        return self.net(x)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=128, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=128, shuffle=False)
final_model = Classifier(**final_config).to(device)
final_optimizer = optim.AdamW(final_model.parameters(), lr=1e-2)
final_criterion = nn.CrossEntropyLoss()
best_path = os.path.join(tmpdir, "final_best.pt")
last_path = os.path.join(tmpdir, "final_last.pt")
best_val_loss = float("inf")
history = []

for epoch in range(10):
    final_model.train(); train_total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        final_optimizer.zero_grad()
        batch_loss = final_criterion(final_model(xb), yb)
        batch_loss.backward(); final_optimizer.step()
        train_total += batch_loss.item() * len(xb)
    final_model.eval(); val_total = 0.0; correct = 0
    with torch.inference_mode():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = final_model(xb)
            val_total += final_criterion(logits, yb).item() * len(xb)
            correct += (logits.argmax(1) == yb).sum().item()
    train_loss = train_total / len(X_train)
    val_loss = val_total / len(X_val)
    val_acc = correct / len(X_val)
    payload = {
        "model_state_dict": final_model.state_dict(),
        "optimizer_state_dict": final_optimizer.state_dict(),
        "epoch": epoch, "config": final_config,
        "best_val_loss": min(best_val_loss, val_loss),
        "class_names": final_class_names,
    }
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        payload["best_val_loss"] = best_val_loss
        torch.save(payload, best_path)
    torch.save(payload, last_path)
    history.append((train_loss, val_loss, val_acc))
    print(f"epoch={epoch:02d} train={train_loss:.3f} val={val_loss:.3f} acc={val_acc:.3f}")

assert os.path.exists(best_path) and os.path.exists(last_path)


In [ ]:
# Simulate a new program: create completely new model and optimizer objects.
best_ckpt = torch.load(best_path, map_location=device, weights_only=False)
restored_best_model = Classifier(**best_ckpt["config"]).to(device)
restored_best_optimizer = optim.AdamW(restored_best_model.parameters(), lr=1e-2)
restored_best_model.load_state_dict(best_ckpt["model_state_dict"])
restored_best_optimizer.load_state_dict(best_ckpt["optimizer_state_dict"])

# Compare against a second independent object loaded from the exact same best state.
expected_best_model = Classifier(**best_ckpt["config"]).to(device)
expected_best_model.load_state_dict(best_ckpt["model_state_dict"])
fixed_batch = X_val[:16].to(device)
expected_best_model.eval(); restored_best_model.eval()
with torch.inference_mode():
    expected_logits = expected_best_model(fixed_batch)
    restored_logits = restored_best_model(fixed_batch)
assert torch.allclose(expected_logits, restored_logits, atol=1e-6)
print("✅ Independent best-model restoration verified")


In [ ]:
# Resume from LAST, not BEST, for two more epochs.
last_ckpt = torch.load(last_path, map_location=device, weights_only=False)
resume_final_model = Classifier(**last_ckpt["config"]).to(device)
resume_final_optimizer = optim.AdamW(resume_final_model.parameters(), lr=1e-2)
resume_final_model.load_state_dict(last_ckpt["model_state_dict"])
resume_final_optimizer.load_state_dict(last_ckpt["optimizer_state_dict"])
resume_start_epoch = last_ckpt["epoch"] + 1
assert resume_start_epoch == 10

resume_losses = []
for epoch in range(resume_start_epoch, resume_start_epoch + 2):
    resume_final_model.train(); total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        resume_final_optimizer.zero_grad()
        loss = final_criterion(resume_final_model(xb), yb)
        loss.backward(); resume_final_optimizer.step()
        total += loss.item() * len(xb)
    resume_losses.append(total / len(X_train))
print("✅ Resumed epochs", resume_start_epoch, "and", resume_start_epoch + 1, resume_losses)


In [ ]:
# Ordered batch inference from the best checkpoint.
ordered_loader = DataLoader(TensorDataset(X_val), batch_size=64, shuffle=False)
restored_best_model.eval()
final_predictions, final_probabilities = [], []
with torch.inference_mode():
    for (xb,) in ordered_loader:
        logits = restored_best_model(xb.to(device))
        final_probabilities.append(torch.softmax(logits, dim=1).cpu())
        final_predictions.append(logits.argmax(dim=1).cpu())
final_predictions = torch.cat(final_predictions)
final_probabilities = torch.cat(final_probabilities)
assert final_predictions.shape == (len(X_val),)
assert final_probabilities.shape == (len(X_val), 4)
assert torch.allclose(final_probabilities.sum(1), torch.ones(len(X_val)), atol=1e-5)
labels = [best_ckpt["class_names"][i] for i in final_predictions[:10].tolist()]
print("First 10 labels:", labels)


In [ ]:
# Device-portability test: the same best artifact must work on CPU.
cpu_ckpt = torch.load(best_path, map_location="cpu", weights_only=False)
cpu_model = Classifier(**cpu_ckpt["config"])
cpu_model.load_state_dict(cpu_ckpt["model_state_dict"])
cpu_model.eval()
with torch.inference_mode():
    cpu_logits = cpu_model(X_val[:4])
assert cpu_logits.device.type == "cpu" and cpu_logits.shape == (4, 4)
print("✅ Checkpoint is CPU-portable")


### Optional challenge — broken 5-class head


In [ ]:
broken_config = dict(final_config, num_classes=5)
five_class_model = Classifier(**broken_config)
four_class_state = cpu_ckpt["model_state_dict"]
five_state = five_class_model.state_dict()
compatible_state = {k: v for k, v in four_class_state.items() if k in five_state and v.shape == five_state[k].shape}
partial = five_class_model.load_state_dict(compatible_state, strict=False)
assert five_class_model.net[-1].weight.shape == (5, 32)
assert set(partial.missing_keys) == {"net.4.weight", "net.4.bias"}
print("✅ Shared backbone loaded; 5-class head left initialized")


## 36. Cheat Sheet


```python
# Weights only
torch.save(model.state_dict(), "model.pt")
state = torch.load("model.pt", map_location=device, weights_only=True)
model = Model(...)
model.load_state_dict(state)
model.to(device).eval()

# Full checkpoint
torch.save({
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": config,
    "best_val_loss": best_val_loss,
}, "checkpoint.pt")

# Restore a trusted local checkpoint
ckpt = torch.load("checkpoint.pt", map_location=device, weights_only=False)
model = Model(**ckpt["config"]).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)
model.load_state_dict(ckpt["model_state_dict"])
optimizer.load_state_dict(ckpt["optimizer_state_dict"])
start_epoch = ckpt["epoch"] + 1

# Inference
model.eval()
with torch.inference_mode():
    logits = model(X.to(device))
    probabilities = torch.softmax(logits, dim=1)
    predictions = logits.argmax(dim=1)

# Binary
probabilities = torch.sigmoid(logits)
predictions = (probabilities >= 0.5).long()
```

### Critical mental models

```text
WEIGHTS ONLY: architecture code + state_dict = usable model
RESUME: model + optimizer + counters + other training state = continuation
INFERENCE: load → eval → inference_mode → same preprocessing → forward → postprocess → CPU/output
```

### Interview quick reference

| Item | Meaning |
|---|---|
| `state_dict` | Named parameter/buffer tensors |
| `torch.save` / `torch.load` | Serialize / restore state |
| `load_state_dict` | Copy state into an existing module |
| `map_location` | Control load device |
| `strict=True` | Exact key/shape compatibility expected |
| `strict=False` | Allow missing/unexpected keys, not shape conflicts |
| `model.eval()` | Evaluation behavior |
| `inference_mode()` | Disable Autograd overhead |
| best / last | Best validation metric / most recent state |

TorchScript is an export/optimization route for some environments; `torch.compile` can optimize execution. Both are awareness topics here—not replacements for correct checkpointing, preprocessing, and inference semantics.


### Checkpoint syntax flashcards (15)


**Get model state?** <details><summary>Answer</summary>`model.state_dict()`</details>


**Load model state?** <details><summary>Answer</summary>`model.load_state_dict(state)`</details>


**Save an object?** <details><summary>Answer</summary>`torch.save(obj, path)`</details>


**Load weights on CPU?** <details><summary>Answer</summary>`torch.load(path, map_location='cpu', weights_only=True)`</details>


**Enter inference mode?** <details><summary>Answer</summary>`with torch.inference_mode():`</details>


**Evaluation behavior?** <details><summary>Answer</summary>`model.eval()`</details>


**Training behavior?** <details><summary>Answer</summary>`model.train()`</details>


**Get optimizer state?** <details><summary>Answer</summary>`optimizer.state_dict()`</details>


**Load optimizer state?** <details><summary>Answer</summary>`optimizer.load_state_dict(state)`</details>


**Next epoch?** <details><summary>Answer</summary>`checkpoint['epoch'] + 1`</details>


**Multiclass prediction?** <details><summary>Answer</summary>`logits.argmax(dim=1)`</details>


**Multiclass probabilities?** <details><summary>Answer</summary>`torch.softmax(logits, dim=1)`</details>


**Binary probabilities?** <details><summary>Answer</summary>`torch.sigmoid(logits)`</details>


**Store device output?** <details><summary>Answer</summary>`output.cpu()`</details>


**Inspect partial load?** <details><summary>Answer</summary>`result.missing_keys`, `result.unexpected_keys`</details>


## Before Moving to `14_pytorch_training_techniques.ipynb`

- [ ] I understand what `state_dict` contains—and that it excludes architecture.
- [ ] I can save, restore, and verify model weights.
- [ ] I can save optimizer state and a full training checkpoint.
- [ ] I can resume from the correct next epoch.
- [ ] I can save best and last checkpoints for different purposes.
- [ ] I understand `map_location` and CPU portability.
- [ ] I understand strict/non-strict loading and shape mismatches.
- [ ] I can replace a classifier head while preserving a backbone.
- [ ] I can perform multiclass, binary, and regression inference.
- [ ] I use both `model.eval()` and `torch.inference_mode()`.
- [ ] I can run ordered, memory-aware batch inference.
- [ ] I understand latency vs throughput.
- [ ] I preserve preprocessing, configuration, and class mappings.
- [ ] I can diagnose missing/unexpected keys and device errors.
- [ ] I treat untrusted serialized checkpoint files as a security risk.

> **Interview synthesis:** deploy the best weights/config/preprocessing artifact; separately retain a full last checkpoint with optimizer and counters so training can resume. Load portably with `map_location`, reconstruct the architecture, verify outputs, and infer with `eval()` plus `inference_mode()`.
